# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset via mlcroissant
dataset = mlc.Dataset(url)

# Access dataset metadata (object attributes)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Publication date: {dataset.metadata.datePublished}")
print(f"Citation: {getattr(dataset.metadata, 'citeAs', '')}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all the record sets and fields for exploration
record_sets = dataset.metadata.recordSets

print(f'Total record sets: {len(record_sets)}')

record_set_ids = []
for rs in record_sets:
    print(f"Record Set name: {rs.name} | @id: {rs.id}")
    fields = getattr(rs, 'fields', [])
    for f in fields:
        print(f"  Field: {getattr(f, 'name', None)} | @id: {getattr(f, 'id', None)} | Data type: {getattr(f, 'dataType', None)}")
    record_set_ids.append(rs.id)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Load all available record sets into dataframes
# If there are multiple, iterate; here we show loading all
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id}, shape: {df.shape}")

# For illustration, pick the first record set for EDA
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id is not None:
    display_columns = dataframes[main_record_set_id].columns.tolist()
    print(f"Columns in {main_record_set_id}:\n", display_columns)
    # Show first five rows
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes to prepare for analysis.

In [ ]:
import numpy as np

# Choose a numeric field (by @id) for analysis -- for this dataset, 
# let's try common candidates such as 'MW [kDa]' or 'num_peptides', etc.

# We'll try to select a column with numeric values
possible_numeric_fields = [
    col for col in dataframes[main_record_set_id].columns
    if any(key in col.lower() for key in ['mw', 'num', 'count', 'abundance', 'peptide', 'coverage'])
]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # E.g. 'MW [kDa]', 'num_peptides'
    print(f"Using {numeric_field} as the numeric field for EDA.")
else:
    raise ValueError("No numeric field found.")

# Convert field values to numeric (if necessary)
df = dataframes[main_record_set_id]
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter for values greater than a threshold
threshold = df[numeric_field].quantile(0.75) if not df[numeric_field].isnull().all() else 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available, e.g., 'description', 'accession', etc.
possible_group_fields = [
    col for col in df.columns if col.lower() in ['description', 'accession', 'gene', 'sample']
]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by {group_field}.")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}, showing first 5 groups:")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

if possible_group_fields:
    # For the top N groups by mean value
    top_n = 10
    grouped_mean = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(top_n)
    plt.figure(figsize=(10,5))
    sns.barplot(x=grouped_mean.index, y=grouped_mean.values)
    plt.title(f'Top {top_n} {group_field} by mean {numeric_field}')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f'Mean {numeric_field}')
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()


## 6. Conclusion
This notebook demonstrates loading, exploring, and performing initial analysis and visualization of the dataset using the Croissant schema and the `mlcroissant` Python library. Key findings:

- The dataset contains detailed mass spectrometry results with protein abundance and modification-related fields.
- We displayed the available fields and record set ids by their `@id`, ensuring reproducibility.
- We filtered and analyzed one numeric field, showing how to normalize values and group by another attribute.
- The approach shown can be extended to more advanced cleaning, joining across record sets, or ML tasks with minor extension.

For in-depth biological or proteomics analysis, refer to dataset documentation and original publication for field semantics.